<a href="https://colab.research.google.com/github/bah862696-coder/DI-Bootcamp/blob/master/DayChallenge_week6_J6.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

## 1. Chargement et inspection des données

In [12]:
# Force reinstallation of specific, stable versions of datasets, huggingface_hub, numpy, and scipy
!pip install datasets==2.16.1 huggingface_hub==1.5.0 numpy==1.26.4 scipy==1.12.0 --force-reinstall -q

ERROR: pip's dependency resolver does not currently take into account all the packages that are installed. This behaviour is the source of the following dependency conflicts.
ipython 7.34.0 requires jedi>=0.16, which is not installed.
google-colab 1.0.0 requires pandas==2.2.2, but you have pandas 3.0.3 which is incompatible.
google-colab 1.0.0 requires requests==2.32.4, but you have requests 2.34.2 which is incompatible.
gcsfs 2025.3.0 requires fsspec==2025.3.0, but you have fsspec 2023.10.0 which is incompatible.
shap 0.52.0 requires numpy>=2, but you have numpy 1.26.4 which is incompatible.
tobler 0.14.0 requires numpy>=2.0, but you have numpy 1.26.4 which is incompatible.
tobler 0.14.0 requires scipy>=1.13, but you have scipy 1.12.0 which is incompatible.
jaxlib 0.7.2 requires numpy>=2.0, but you have numpy 1.26.4 which is incompatible.
jaxlib 0.7.2 requires scipy>=1.13, but you have scipy 1.12.0 which is incompatible.
opencv-python 4.13.0.92 requires numpy>=2; python_version >= "3.

In [4]:
import os
import shutil
from datasets import load_dataset

# Set environment variable to potentially workaround HfUriError
os.environ['HF_HUB_ENABLE_HF_SCAN'] = 'False'

# Clear Hugging Face cache to ensure fresh download and resolve potential issues
# This step is included again in case the reinstallation changed cache paths or behavior.
cache_dir = os.path.expanduser("~/.cache/huggingface")
if os.path.exists(cache_dir):
    print(f"Clearing Hugging Face cache at {cache_dir}")
    shutil.rmtree(cache_dir)
else:
    print(f"Hugging Face cache directory not found at {cache_dir}")

# Load the tweet_eval dataset with the 'sentiment' configuration
dataset = load_dataset('tweet_eval', 'sentiment')

# Display the data distribution (sizes of train, validation, test splits)
print("\nDataset splits distribution:")
for split, data in dataset.items():
    print(f"  {split}: {len(data)} examples")

# Check the class distribution for the 'train' split
print("\nClass distribution for the 'train' split:")
# The labels are 'negative', 'neutral', 'positive'
# tweet_eval sentiment labels are 0: negative, 1: neutral, 2: positive
label_names = dataset['train'].features['label'].names

for i, name in enumerate(label_names):
    count = sum(1 for label in dataset['train']['label'] if label == i)
    print(f"  Label {i} ({name}): {count} examples")

# Save two example tweets per label for later visualization
example_tweets = {}
for i, name in enumerate(label_names):
    examples = [text for text, label in zip(dataset['train']['text'], dataset['train']['label']) if label == i][:2]
    example_tweets[name] = examples

print("\nExample tweets per label (for later visualization):")
for label, tweets in example_tweets.items():
    print(f"  {label}:\n    - '{tweets[0]}'\n    - '{tweets[1]}'")

Clearing Hugging Face cache at /root/.cache/huggingface


Generating train split:   0%|          | 0/45615 [00:00<?, ? examples/s]

Generating test split:   0%|          | 0/12284 [00:00<?, ? examples/s]

Generating validation split:   0%|          | 0/2000 [00:00<?, ? examples/s]


Dataset splits distribution:
  train: 45615 examples
  test: 12284 examples
  validation: 2000 examples

Class distribution for the 'train' split:
  Label 0 (negative): 7093 examples
  Label 1 (neutral): 20673 examples
  Label 2 (positive): 17849 examples

Example tweets per label (for later visualization):
  negative:
    - 'So disappointed in wwe summerslam! I want to see john cena wins his 16th title'
    - 'That sucks if you have to take the SATs tomorrow'
  neutral:
    - '"Ben Smith / Smith (concussion) remains out of the lineup Thursday, Curtis #NHL #SJ"'
    - 'Sorry bout the stream last night I crashed out but will be on tonight for sure. Then back to Minecraft in pc tomorrow night.'
  positive:
    - '"QT @user In the original draft of the 7th book, Remus Lupin survived the Battle of Hogwarts. #HappyBirthdayRemusLupin"'
    - '@user Alciato: Bee will invest 150 million in January, another 200 in the Summer and plans to bring Messi by 2017"'


## 2. Pipeline de tokenisation

In [15]:
import sys

# Consolidate all necessary installations to ensure compatibility and fresh loading after restart

# First, ensure system-level build tools for Rust are available
# Removed > /dev/null to see output and verify installation
!apt-get update -qq
!apt-get install -y rustc cargo

# Upgrade pip and wheel for general robustness
!pip install --upgrade pip wheel -q

# Install setuptools-rust, which helps Python build Rust extensions
!pip install setuptools-rust -q

# Install main Hugging Face libraries, letting pip resolve dependencies for tokenizers and huggingface_hub
# Using a slightly more recent, yet stable, version of transformers (v4.38.2 released Feb 2024)
# This should pull compatible tokenizers and huggingface_hub versions.
!pip install transformers==4.38.2 -q

# Force reinstallation of datasets, numpy, and scipy to ensure consistent versions after core HF libs
# Keep these pinned as they were identified as sources of compatibility issues before
!pip install datasets==2.16.1 numpy==1.26.4 scipy==1.12.0 --force-reinstall -q

print("Installation attempt complete. Please RESTART YOUR RUNTIME now (Runtime -> Restart runtime) for changes to take effect.")

  Installing build dependencies ... done
  Getting requirements to build wheel ... done
  Preparing metadata (pyproject.toml) ... done
  error: subprocess-exited-with-error
  
  × Building wheel for tokenizers (pyproject.toml) did not run successfully.
  │ exit code: 1
  ╰─> No available output.
  
  note: This error originates from a subprocess, and is likely not a problem with pip.
  ERROR: Failed building wheel for tokenizers
error: failed-wheel-build-for-install

× Failed to build installable wheels for some pyproject.toml based projects
╰─> tokenizers
  Installing build dependencies ... done
  Getting requirements to build wheel ... done
  Preparing metadata (pyproject.toml) ... done
  error: subprocess-exited-with-error
  
  × Building wheel for tokenizers (pyproject.toml) did not run successfully.
  │ exit code: 1
  ╰─> No available output.
  
  note: This error originates from a subprocess, and is likely not a problem with pip.
  ERROR: Failed building wheel for tokenizers
erro

In [6]:
from transformers import AutoTokenizer

# Initialize AutoTokenizer with distilbert-base-uncased
tokenizer = AutoTokenizer.from_pretrained('distilbert-base-uncased')

# Define the preprocessing function
def preprocess_function(examples):
    # Truncate/pad tweets to 128 tokens, return input_ids, attention_mask
    return tokenizer(examples['text'], truncation=True, padding='max_length', max_length=128)

# Map the preprocessing function to the entire dataset
tokenized_dataset = dataset.map(preprocess_function, batched=True)

# Remove the original 'text' column as it's no longer needed after tokenization
tokenized_dataset = tokenized_dataset.remove_columns(["text"])

# Rename the 'label' column to 'labels' to match the expected format for Hugging Face Trainer
tokenized_dataset = tokenized_dataset.rename_column("label", "labels")

# Set the format to "torch" (or "tf" if using TensorFlow)
tokenized_dataset.set_format("torch")

# Shuffle the training set
tokenized_dataset["train"] = tokenized_dataset["train"].shuffle(seed=42)

# Display a sample of the tokenized dataset
print("\nSample of tokenized dataset (training split):")
print(tokenized_dataset["train"][0])

print("\nDataset features after tokenization:")
print(tokenized_dataset["train"].features)

ImportError: cannot import name 'DryRunError' from 'huggingface_hub.errors' (/usr/local/lib/python3.12/dist-packages/huggingface_hub/errors.py)